# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. We will:
- Load and inspect the dataset metadata and available record sets
- Extract and explore the data
- Perform basic exploratory data analysis (EDA)
- Visualize key aspects of the data

### Dataset Source
The dataset is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import pprint

# Define the Croissant metadata file URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Print high-level metadata
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n")
print(f"Dataset description: {metadata.description}\n")
print(f"Published on: {getattr(metadata, 'datePublished', None)}\n")

## 2. Data Overview
Review available record sets, their `@id`s, fields/columns, and a metadata summary of each.

In [ ]:
# List all available record sets and their ids
record_sets = dataset.metadata.record_sets
print("Available record sets and their @id:\n")
for rs in record_sets:
    print(f"- name: {rs.name} | @id: {rs.id_} | description: {getattr(rs, 'description', None)}")

# Let's print details about the first record set (the main data table)
main_record_set = record_sets[0] if record_sets else None

if main_record_set:
    print("\n--- Main Record Set Info ---\n")
    print(f"@id: {main_record_set.id_}")
    print(f"Name: {main_record_set.name}")
    print(f"Description: {getattr(main_record_set, 'description', None)}\n")
    
    print("Available fields (columns) in this record set:\n")
    for field in main_record_set.fields:
        print(f"- {getattr(field, 'name', '')} (@id: {field.id_}, dtype: {getattr(field, 'data_type', None)})")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. All `record_set`, `field`, and `column` references use their `@id`.

In [ ]:
# We'll extract the data using the main record set @id
dataframes = {}

main_record_set_id = main_record_set.id_ if main_record_set else None
if main_record_set_id:
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    dataframes[main_record_set_id] = df

    print(f"Extracted columns in main DataFrame (@id: {main_record_set_id}):\n{df.columns.tolist()}")
    display_cols = df.columns[:10] if len(df.columns) > 10 else df.columns
    display(df[display_cols].head(5))
else:
    print("No record set found in the dataset.")

## 4. Exploratory Data Analysis (EDA)

Let's investigate a numeric column such as `TotalPeptides` or `MW[kDa]` (for example). We will:
- Filter to high-mass proteins
- Normalize the values
- (If possible) group by a categorical variable

All field/column access is based on their `@id` (i.e., DataFrame column names).

In [ ]:
# Find a numeric and a grouping field from the record set
if main_record_set and main_record_set.fields:
    # Find plausible numeric field
    numeric_field_id = None
    group_field_id = None
    for field in main_record_set.fields:
        if not numeric_field_id and getattr(field, 'data_type', None) in ('Float', 'Integer', 'Number'):
            numeric_field_id = field.id_
        if not group_field_id and getattr(field, 'data_type', None) == 'Text':
            group_field_id = field.id_

    print(f"Using numeric field: {numeric_field_id}")
    print(f"Using group field: {group_field_id}")
else:
    numeric_field_id = None
    group_field_id = None

df = dataframes.get(main_record_set_id)
if df is not None and numeric_field_id in df.columns:
    # Drop rows with missing values to avoid issues
    filtered_df = df[df[numeric_field_id].astype(float) > 10].copy()
    print(f"Filtered records with {numeric_field_id} > 10 (N={len(filtered_df)}):\n")
    display(filtered_df[[numeric_field_id]].head(5))

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(5))
    
    # Group by a categorical field if it exists and has a reasonable number of categories
    if group_field_id and group_field_id in df.columns:
        n_keys = filtered_df[group_field_id].nunique()
        if 1 < n_keys < 51:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped average {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print(f"'{group_field_id}' not used for grouping (not in 2..50 unique values, actual: {n_keys}).")
    else:
        print("No suitable group field found.")
else:
    print(f"Cannot perform EDA; missing main data or field '{numeric_field_id}'.")

## 5. Visualization

Visualize the distribution of the selected numeric variable. For example, plot a histogram of the protein molecular weights or peptide counts.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna().astype(float), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()
else:
    print(f"No data to plot histogram for '{numeric_field_id}'.")

# Boxplot by group if group field makes sense
if group_field_id and df is not None and group_field_id in df.columns and numeric_field_id in df.columns:
    n_keys = df[group_field_id].nunique()
    # Only plot if reasonable group size
    if 1 < n_keys < 15:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.show()

## 6. Conclusion

In this notebook, we loaded, inspected, and performed basic EDA on the FAIR² dataset of extracellular vesicle protein abundance. The `mlcroissant` library allows for schema-driven dataset exploration that makes code robust to schema updates or metadata changes by consistently referencing objects by their `@id`. You can further utilize the Croissant schema for programmatic, reproducible FAIR data processing and ML-ready workflows.